In [ ]:
import re, json, os
from typing import List, Dict

from dotenv import load_dotenv
load_dotenv()

#os.environ['OPENAI_API_KEY'] = ""


input_file = "../dataset/original_formatted/train.json"
output_file = "../dataset/original_english_formatted/train.json"


from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    request_timeout=60,
)


In [10]:
def normalize_speaker_labels(raw_dialogue: str) -> str:
    """
    줄 시작의 '화자1:' / '화자2:' 만 'Speaker1:' / 'Speaker2:' 로 치환.
    본문 내 '화자1', '화자2' 텍스트는 변경하지 않음.
    """
    s = re.sub(r'(?m)^\s*화자1\s*:', 'Speaker1:', raw_dialogue)
    s = re.sub(r'(?m)^\s*화자2\s*:', 'Speaker2:', s)
    return s

def parse_dialogue_to_turns(dialogue_str: str) -> List[Dict[str, str]]:
    """
    "Speaker1: ..." / "Speaker2: ..." 라인들을 [{'speaker': 'Speaker1', 'text': '...'}, ...] 로 변환
    """
    turns = []
    for line in dialogue_str.splitlines():
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^(Speaker1|Speaker2)\s*:\s*(.*)$', line)
        if m:
            speaker, text = m.group(1), m.group(2).strip()
            turns.append({"speaker": speaker, "text": text})
    return turns


In [11]:
SYSTEM_PROMPT = (
    "You are a precise translator. Read the short dialogue context in Korean, "
    "then translate ONLY the final utterance into natural, fluent English.\n"
    "Rules: Output ONLY the English translation, no explanations.\n\n"
    "Here are two examples:\n\n"
    "Example 1:\n"
    "Context:\n"
    "Speaker1: 안녕하세요! 오늘 날씨가 참 좋네요.\n"
    "Speaker2: 네, 그래서 산책하려고요.\n"
    "Final utterance (Korean):\n"
    "Speaker1: 좋은 생각이네요, 저도 같이 갈까요?\n"
    "Output:\n"
    "That's a great idea, shall I join you?\n\n"
    "Example 2:\n"
    "Context:\n"
    "Speaker1: 주말에 뭐 하실 계획이세요?\n"
    "Speaker2: 친구들이랑 등산 가려고요.\n"
    "Speaker1: 어디로 가세요?\n"
    "Speaker2: 북한산이요.\n"
    "Final utterance (Korean):\n"
    "Speaker1: 저도 거기 정말 좋아해요!\n"
    "Output:\n"
    "I really like that place too!\n\n"
    "Now, follow the same style for the next input."
)

# === 단일 발화 번역용 메시지 빌더 (신규) ===
def build_messages_for_single_turn(context_turns: List[Dict[str, str]],
                                   final_turn: Dict[str, str],
                                   k_context: int = 4):
    """
    - context_turns: 직전 컨텍스트 발화들(길이 <= k_context)
    - final_turn: 지금 번역할 타겟 발화 {'speaker':..., 'text':...}
    반환: OpenAI(Chat) 포맷 messages
    """
    # 컨텍스트 문자열
    ctx_lines = [f"{t['speaker']}: {t['text']}" for t in context_turns]
    context_block = "\n".join(ctx_lines) if ctx_lines else "(no previous context)"

    user_prompt = (
        f"Context (up to last {k_context} utterances):\n"
        f"{context_block}\n\n"
        f"Final utterance (Korean):\n"
        f"{final_turn['speaker']}: {final_turn['text']}\n\n"
        f"Task: Translate the FINAL utterance to English. "
        f"Return ONLY the translation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]

# === 전체 대화를 '한 발화씩' 순차 번역하고, 영어 대화 문자열로 반환 (신규) ===
def translate_dialogue_all_utterances(dialogue_str: str, k_context: int = 4,
                                      max_tries: int = 3, base_wait: float = 1.5) -> str:
    """
    대화를 순회하며 매 차례:
      - 직전 최대 k_context개의 발화를 컨텍스트로 제공
      - 현재(마지막) 발화만 번역
    최종적으로 "SpeakerX: <영문>" 줄들의 조합으로 하나의 문자열을 반환.
    """
    norm = normalize_speaker_labels(dialogue_str)
    turns = parse_dialogue_to_turns(norm)
    if not turns:
        return ""

    translated_lines = []

    for i, turn in enumerate(turns):
        # 직전 k개 컨텍스트
        ctx = turns[max(0, i - k_context): i]
        messages = build_messages_for_single_turn(ctx, turn, k_context=k_context)

        # 간단 재시도
        last_err = None
        for t in range(max_tries):
            try:
                resp = llm.invoke(messages)
                eng = (resp.content or "").strip()
                translated_lines.append(f"{turn['speaker']}: {eng}")
                break
            except Exception as e:
                last_err = e
                time.sleep(base_wait * (t + 1))
        else:
            # 모든 재시도 실패 시 빈 줄/오류 마킹
            translated_lines.append(f"{turn['speaker']}: ")
    
    return "\n".join(translated_lines)

In [ ]:
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

total_utterances = 0
for rec in data:
    dlg = rec.get("dialogue", "") or ""
    if dlg.strip():
        norm = normalize_speaker_labels(dlg)
        turns = parse_dialogue_to_turns(norm)
        total_utterances += len(turns)

print(f"Total utterances to translate: {total_utterances}")

Total utterances to translate: 17110


## dialogue 항목 번역

In [ ]:
import os, json, time
from tqdm.auto import tqdm

K_CONTEXT = 4
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 데이터 로딩
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)
assert isinstance(data, list), "입력 JSON은 리스트여야 합니다."

processed = []
n_err = 0

for rec in tqdm(data, desc="Translating all utterances (k=4)", ncols=90):
    dlg = rec.get("dialogue", "") or ""
    try:
        if dlg.strip():
            # 전체 발화를 순차 번역하고, 번역 결과를 dialogue에 덮어쓰기
            translated_dialogue = translate_dialogue_all_utterances(dlg, k_context=K_CONTEXT)
            rec["dialogue"] = translated_dialogue
        else:
            rec["dialogue"] = ""
    except Exception as e:
        n_err += 1
        # 실패 시 원문 유지하거나 빈 문자열로 처리(여기선 원문 유지 선택)
        rec["dialogue"] = dlg
        rec["translation_error"] = str(e)
    processed.append(rec)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(processed, f, ensure_ascii=False, indent=2)

print(f"Done. total={len(processed)}, errors={n_err}, saved -> {output_file}")

Translating all utterances (k=4):   1%|▏                | 6/506 [02:05<2:42:58, 19.56s/it]

## Question 항목 번역 

In [ ]:
# === 1) SYSTEM PROMPT for 'output' translation (few-shot) ===
SYSTEM_PROMPT_OUTPUT = (
    "You are a meticulous translator. Translate the Korean summary found in a dataset's "
    "'output' field into clear, neutral English.\n"
    "Rules:\n"
    "- Preserve names, IDs (e.g., SD2000691), numbers, and line breaks.\n"
    "- Do NOT add, remove, or invent information.\n"
    "- Output ONLY the translated text, no explanations.\n\n"
    "Example 1:\n"
    "Input:\n"
    "두 화자는 이 대화에서 병원 진료와 자녀를 위한 건강식품에 대해 말했습니다. "
    "SD2000691은 자녀들에게 유산균과 종합 비타민을 챙겨준다고 말했습니다.\n"
    "Output:\n"
    "The two speakers discussed medical visits and health supplements for their children. "
    "SD2000691 said they give their children probiotics and a multivitamin.\n\n"
    "Example 2:\n"
    "Input:\n"
    "SD2000694는 스트레스를 풀기 위해 매운 음식을 좋아한다고 말했습니다.\n"
    "Output:\n"
    "SD2000694 said they like spicy food to relieve stress.\n\n"
    "Now, translate the next summary following the same style."
)


In [ ]:
# === 2) Builders & Runner for 'output' translation ===
import time, json
from typing import List, Dict

def build_messages_for_output(output_text: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_OUTPUT},
        {"role": "user",   "content": (output_text or "").strip()},
    ]

def translate_output_block(output_text: str, max_tries: int = 3, base_wait: float = 1.5) -> str:
    last_err = None
    msgs = build_messages_for_output(output_text)
    for t in range(max_tries):
        try:
            resp = llm.invoke(msgs)
            return (resp.content or "").strip()
        except Exception as e:
            last_err = e
            time.sleep(base_wait * (t + 1))
    # 실패 시 원문 유지
    return output_text


In [ ]:
# === 3) Reload -> translate 'output' -> add 'output_en' -> save back to same file ===
from tqdm.auto import tqdm

# assumes: output_file already defined (e.g., "../dataset/original_english_formatted/ellipsis_recovered_train.json")
with open(output_file, "r", encoding="utf-8") as f:
    ds = json.load(f)
assert isinstance(ds, list), "Output dataset must be a list."

err_cnt = 0
for rec in tqdm(ds, desc="Translating 'output' field", ncols=90):
    out_text = rec.get("output", "")
    if isinstance(out_text, str) and out_text.strip():
        try:
            rec["output_en"] = translate_output_block(out_text)
        except Exception as e:
            err_cnt += 1
            rec["output_en"] = out_text
            rec["output_translation_error"] = str(e)
    else:
        rec["output_en"] = ""

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ds, f, ensure_ascii=False, indent=2)

print(f"Done translating 'output'. errors={err_cnt}. Saved -> {output_file}")


## 키워드 번역

In [ ]:
# === 4) SYSTEM PROMPT for subject_keyword list translation (JSON array output) ===
SYSTEM_PROMPT_SUBJECT_KEYWORDS = (
    "You are a careful translator. Translate each Korean subject keyword into concise English (1–3 words).\n"
    "Rules:\n"
    "- Preserve list length and order exactly.\n"
    "- Output ONLY a JSON array of strings (e.g., [\"Food\", \"Health\", \"Examination\"]).\n"
    "- Do NOT include explanations.\n\n"
    "Example:\n"
    "Input:\n"
    "[\"식품\", \"건강\", \"검사\"]\n"
    "Output:\n"
    "[\"Food\", \"Health\", \"Examination\"]\n\n"
    "Now translate the next list."
)


In [ ]:
# === 5) Builders & Runner for subject_keyword translation ===
def build_messages_for_subject_keywords(keywords: List[str]):
    # 모델이 그대로 복사/번역하기 쉽게 JSON 배열 문자열로 제공
    user_content = json.dumps([str(k) for k in (keywords or [])], ensure_ascii=False)
    return [
        {"role": "system", "content": SYSTEM_PROMPT_SUBJECT_KEYWORDS},
        {"role": "user",   "content": user_content},
    ]

def translate_single_keyword_kw2en(keyword: str) -> str:
    # 폴백: 개별 키워드 1~3단어 번역
    prompt = (
        "Translate the following Korean keyword into concise English (1–3 words). "
        "Output ONLY the English phrase.\n\n"
        f"Keyword: {keyword}"
    )
    msgs = [{"role":"system","content":"You translate single keywords precisely and concisely."},
            {"role":"user","content":prompt}]
    resp = llm.invoke(msgs)
    return (resp.content or "").strip()

def translate_subject_keywords_list(keywords: List[str], max_tries: int = 3, base_wait: float = 1.2) -> List[str]:
    if not isinstance(keywords, list):
        return []
    msgs = build_messages_for_subject_keywords(keywords)
    last_err = None
    for t in range(max_tries):
        try:
            resp = llm.invoke(msgs)
            txt = (resp.content or "").strip()
            arr = json.loads(txt)
            if isinstance(arr, list) and len(arr) == len(keywords):
                # 모든 항목을 문자열로 캐스팅
                return [str(x) for x in arr]
            # 길이/형식 불일치 → 폴백
            break
        except Exception as e:
            last_err = e
            time.sleep(base_wait * (t + 1))
    # 폴백: 항목별 번역
    return [translate_single_keyword_kw2en(k) for k in keywords]


In [ ]:
sk_err = 0
for rec in tqdm(ds, desc="Translating 'subject_keyword' list", ncols=90):
    kws = rec.get("subject_keyword", [])
    if isinstance(kws, list) and len(kws) > 0:
        try:
            rec["subject_keyword_en"] = translate_subject_keywords_list(kws)
        except Exception as e:
            sk_err += 1
            rec["subject_keyword_en"] = []
            rec["subject_keyword_translation_error"] = str(e)
    else:
        rec["subject_keyword_en"] = []

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ds, f, ensure_ascii=False, indent=2)

print(f"Done translating 'subject_keyword'. errors={sk_err}. Saved -> {output_file}")

## speaker_map 항목 수정

In [ ]:
# === Rename keys in speaker_map: 화자1/화자2 -> speaker1/speaker2 (case-normalize), save back ===
import json, re
from collections import OrderedDict
from tqdm.auto import tqdm

# assumes: output_file already defined and points to the JSON you want to modify
with open(output_file, "r", encoding="utf-8") as f:
    ds = json.load(f)
assert isinstance(ds, list), "Dataset must be a list of records."

def remap_speaker_keys(smap: dict):
    """
    Convert keys:
      - '화자1' -> 'speaker1', '화자2' -> 'speaker2'
      - 'Speaker1'/'Speaker2' -> 'speaker1'/'speaker2' (normalize case)
    Leaves other keys unchanged. Preserves order. Avoids overwriting by suffixing __dupN if needed.
    """
    new_map = OrderedDict()
    changes = 0
    for k, v in smap.items():
        new_k = k
        m_ko = re.fullmatch(r'화자\s*([0-9]+)', k) or re.fullmatch(r'화자([0-9]+)', k)
        m_en = re.fullmatch(r'[sS]peaker\s*([0-9]+)', k) or re.fullmatch(r'[sS]peaker([0-9]+)', k)
        if m_ko:
            new_k = f"speaker{m_ko.group(1)}"
        elif m_en:
            new_k = f"speaker{m_en.group(1)}"  # normalize case

        if new_k != k:
            changes += 1

        # handle potential collision safely
        if new_k in new_map and new_map[new_k] != v:
            i = 2
            alt = f"{new_k}__dup{i}"
            while alt in new_map:
                i += 1
                alt = f"{new_k}__dup{i}"
            new_map[alt] = v
        else:
            new_map[new_k] = v
    return new_map, changes

records_changed = 0
keys_changed = 0

for rec in tqdm(ds, desc="Renaming keys in speaker_map", ncols=90):
    sm = rec.get("speaker_map", None)
    if isinstance(sm, dict) and sm:
        new_sm, chg = remap_speaker_keys(sm)
        if chg > 0:
            rec["speaker_map"] = new_sm
            records_changed += 1
            keys_changed += chg

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ds, f, ensure_ascii=False, indent=2)

print(f"Done. records_changed={records_changed}, keys_changed={keys_changed}, saved -> {output_file}")
